In [1]:
import pandas as pd
import numpy as np

In [2]:
df= pd.read_csv("../raw_data/amazon_india_2017.csv")

In [290]:
df.head()

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
0,TXN_2017_00000001,17-01-2017,CUST_2016_00011375,PROD_000297,Vivo V7+ 16GB Black,Electronics,Smartphones,Vivo,25367.39,0.00,...,FALSE,NaN,NaN,Delivered,1,2017,1,0.23,True,4.6
1,TXN_2017_00000002,2017-01-28,CUST_2017_00014565,PROD_000193,Motorola Moto G4 32GB White,Electronics,Smartphones,Motorola,16997.28,9.98,...,False,NaN,3.0,Returned,1,2017,1,0.16,True,3.5
2,TXN_2017_00000003,2017-01-30,CUST_2016_00011649,PROD_000073,Xiaomi Redmi 2 16GB White,Electronics,Smartphones,Xiaomi,47409.63,0.00,...,False,NaN,3.5,Delivered,1,2017,1,0.23,True,4.2
3,TXN_2017_00000004,2017-01-04,CUST_2016_00003484,PROD_000047,Samsung Galaxy J7 64GB Black,Electronics,Smartphones,Samsung,"₹45,555.75",0.00,...,False,NaN,4.5,Delivered,1,2017,1,0.24,True,4.4
4,TXN_2017_00000005,2017-01-07,CUST_2015_00009936,PROD_000136,Samsung Galaxy S7 Edge 16GB Blue,Electronics,Smartphones,Samsung,100190.24,0.00,...,False,NaN,5.0,Delivered,1,2017,1,0.16,True,3.4


In [291]:
df["delivery_charges"].isna().sum(), len(df)

(np.int64(6189), 77385)

In [292]:
df["delivery_charges"].describe()

count    71196.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: delivery_charges, dtype: float64

In [293]:
df.drop(columns=["delivery_charges"], inplace=True)

In [294]:
df.shape

(77385, 33)

In [295]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77385 entries, 0 to 77384
Data columns (total 33 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   transaction_id          77385 non-null  object 
 1   order_date              77385 non-null  object 
 2   customer_id             77385 non-null  object 
 3   product_id              77385 non-null  object 
 4   product_name            77385 non-null  object 
 5   category                77385 non-null  object 
 6   subcategory             77385 non-null  object 
 7   brand                   77385 non-null  object 
 8   original_price_inr      77385 non-null  object 
 9   discount_percent        77385 non-null  float64
 10  discounted_price_inr    77385 non-null  float64
 11  quantity                77385 non-null  int64  
 12  subtotal_inr            77385 non-null  float64
 13  final_amount_inr        77385 non-null  float64
 14  customer_city           77385 non-null

In [296]:
df.columns

Index(['transaction_id', 'order_date', 'customer_id', 'product_id',
       'product_name', 'category', 'subcategory', 'brand',
       'original_price_inr', 'discount_percent', 'discounted_price_inr',
       'quantity', 'subtotal_inr', 'final_amount_inr', 'customer_city',
       'customer_state', 'customer_tier', 'customer_spending_tier',
       'customer_age_group', 'payment_method', 'delivery_days',
       'delivery_type', 'is_prime_member', 'is_festival_sale', 'festival_name',
       'customer_rating', 'return_status', 'order_month', 'order_year',
       'order_quarter', 'product_weight_kg', 'is_prime_eligible',
       'product_rating'],
      dtype='object')

Question 1
Your dataset contains order_date in multiple formats: 'DD/MM/YYYY', 'DD-MM-YY', 'YYYY-MM-DD', and some invalid entries like '32/13/2020'. Clean and standardize all dates to 'YYYY-MM-DD' format, handling invalid dates appropriately.


In [297]:
df["order_date"].head(20)

0     17-01-2017
1     2017-01-28
2     2017-01-30
3     2017-01-04
4     2017-01-07
5     2017-01-14
6     01/14/2017
7     2017-01-19
8     2017-01-24
9     2017-01-13
10    2017-01-02
11    2017-01-01
12    2017-01-26
13    18-01-2017
14    2017-01-27
15    2017-01-08
16    2017-01-27
17    2017-01-06
18    2017-01-04
19    2017-01-13
Name: order_date, dtype: object

In [298]:
df["order_date"]=(
    df["order_date"]
    .str.replace(" ","", regex=False)
    .str.replace("/","-",regex=False)
)

parts= df["order_date"].str.split("-", expand=True)
year_last=parts[2].str.len()==4
df.loc[year_last, "order_date"]= (parts[2]+"-"+parts[0]+"-"+parts[1])

parts=df["order_date"].str.split("-", expand=True)
mask = parts[1].astype(int)>12
df.loc[mask, "order_date"]= (parts[0]+"-"+parts[2]+"-"+parts[1])
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")


In [299]:
df["order_date"].min(), df["order_date"].max()

(Timestamp('2017-01-01 00:00:00'), Timestamp('2017-12-31 00:00:00'))

In [300]:
df["order_date"].isna().sum()

np.int64(0)

Question 2
The original_price_inr column contains mixed data types: numeric values, text with '₹' symbols, comma separators ('₹1,25,000'), and some entries like 'Price on Request'. Clean this column to contain only numeric values in Indian Rupees. 


In [301]:
df["original_price_inr"] = df["original_price_inr"].astype(str)

df["original_price_inr"] = df["original_price_inr"].str.replace("₹", "", regex=False)

df["original_price_inr"] = df["original_price_inr"].str.replace(",", "", regex=False)

df["original_price_inr"] = pd.to_numeric(df["original_price_inr"], errors="coerce")

In [302]:
df["original_price_inr"].unique()[:20]

array([ 25367.39,  16997.28,  47409.63,  45555.75, 100190.24,  23773.66,
        48131.09,  97392.23,  31992.51,  26170.89,  55223.29, 142435.6 ,
       172043.77,  27630.74,  52356.86,  17974.52,  26822.66,       nan,
        14094.89,  27999.78])

In [303]:
df["original_price_inr"].dtypes

dtype('float64')

Question 3
Customer ratings appear in various formats: '5.0', '4 stars', '3/5', '2.5/5.0', and some missing values. Standardize all ratings to numeric scale 1.0-5.0, handling inconsistent formats and missing values strategically.


In [304]:
df["customer_rating"]=df["customer_rating"].astype(str)
df["customer_rating"]= df["customer_rating"].str.replace("stars","",regex=False)
df["customer_rating"]= df["customer_rating"].str.split("/").str[0]
df["customer_rating"]= pd.to_numeric(df["customer_rating"],errors="coerce")


In [305]:
df["customer_rating"].describe()

count    53793.000000
mean         4.316445
std          0.569389
min          3.000000
25%          4.000000
50%          4.500000
75%          5.000000
max          5.000000
Name: customer_rating, dtype: float64

In [306]:
df["customer_rating"].value_counts().head(10)

customer_rating
4.5    17673
5.0    13977
4.0    13632
3.5     5440
3.0     3071
Name: count, dtype: int64

In [307]:
df["customer_rating"].isna().sum()

np.int64(23592)

In [308]:
df["customer_rating"].value_counts(dropna=False)

customer_rating
NaN    23592
4.5    17673
5.0    13977
4.0    13632
3.5     5440
3.0     3071
Name: count, dtype: int64

Question 4
The customer_city column has inconsistent naming: 'Bangalore/Bengaluru', 'Mumbai/Bombay', 'Delhi/New Delhi', along with spelling errors and case variations. Standardize all city names and handle geographical variations.


In [309]:
df["customer_city"]= df["customer_city"].str.strip().str.lower()

In [310]:
df["customer_city"].unique()

array(['pune', 'chandigarh', 'ludhiana', 'bangalore', 'mumbai', 'kanpur',
       'patna', 'bhubaneswar', 'delhi', 'nagpur', 'indore', 'ahmedabad',
       'chennai', 'kochi', 'kolkata', 'gorakhpur', 'visakhapatnam',
       'hyderabad', 'lucknow', 'varanasi', 'bengaluru', 'moradabad',
       'jaipur', 'surat', 'coimbatore', 'vadodara', 'meerut', 'aligarh',
       'chenai', 'bareilly', 'saharanpur', 'allahabad', 'bengalore',
       'calcutta', 'bombay', 'mumba', 'madras', 'new delhi', 'delhi ncr',
       'banglore'], dtype=object)

In [311]:
city_map = {
    "bengaluru": "bangalore",
    "bengalore": "bangalore",
    "banglore": "bangalore",

    "chenai": "chennai",
    "madras": "chennai",

    "calcutta": "kolkata",

    "bombay": "mumbai",
    "mumba": "mumbai",

    "new delhi": "delhi",
    "delhi ncr": "delhi"
}

In [312]:
df["customer_city"] = df["customer_city"].replace(city_map)

In [313]:
df["customer_city"] = df["customer_city"].str.title()

In [314]:
df["customer_city"].value_counts().head()


customer_city
Mumbai       11867
Delhi        10232
Bangalore     8731
Chennai       7170
Kolkata       5452
Name: count, dtype: int64

In [315]:
df["customer_city"].unique()


array(['Pune', 'Chandigarh', 'Ludhiana', 'Bangalore', 'Mumbai', 'Kanpur',
       'Patna', 'Bhubaneswar', 'Delhi', 'Nagpur', 'Indore', 'Ahmedabad',
       'Chennai', 'Kochi', 'Kolkata', 'Gorakhpur', 'Visakhapatnam',
       'Hyderabad', 'Lucknow', 'Varanasi', 'Moradabad', 'Jaipur', 'Surat',
       'Coimbatore', 'Vadodara', 'Meerut', 'Aligarh', 'Bareilly',
       'Saharanpur', 'Allahabad'], dtype=object)

Question 5
Boolean columns (is_prime_member, is_prime_eligible, is_festival_sale) contain mixed values: True/False, Yes/No, 1/0, Y/N, and some missing entries. Convert all boolean columns to consistent True/False format.


In [316]:
bool_candidates=[]
bool_values={"true", "false","y","n","yes", "no","0","1"}

for col in df.columns:
    vals= set(df[col].astype(str). str.lower().dropna().unique())
    if vals & bool_values:
        bool_candidates.append(col)
bool_candidates



['quantity',
 'delivery_days',
 'is_prime_member',
 'is_festival_sale',
 'order_month',
 'order_quarter',
 'is_prime_eligible']

In [317]:
boolean_cols = ["is_prime_member", "is_prime_eligible", "is_festival_sale"]

bool_map = {
    "true": True,
    "false": False,
    "yes": True,
    "no": False,
    "y": True,
    "n": False,
    "1": True,
    "0": False
}

for col in boolean_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(bool_map)
    )

In [318]:
df[boolean_cols].value_counts(dropna=False)

is_prime_member  is_prime_eligible  is_festival_sale
False            True               False               39700
                                    True                18382
                 False              False                8986
                                    True                 4251
True             True               False                3341
                                    True                 1587
                 False              False                 804
                                    True                  334
Name: count, dtype: int64

Question 6
Product categories have variations: 'Electronics/Electronic/ELECTRONICS/Electronics & Accessories'. Standardize category names across the dataset and ensure consistent naming conventions.


In [3]:
df["category"].value_counts().head(20)

category
Electronics                  77305
Electronicss                    28
ELECTRONICS                     25
Electronic                      16
Electronics & Accessories       11
Name: count, dtype: int64

In [4]:
df["category"] = df["category"].str.strip().str.lower()


In [5]:
category_map = {
    "electronic": "electronics",
    "electronics": "electronics",
    "electronics & accessories": "electronics",
    "electronicss": "electronics"
}

In [6]:
df["category"] = df["category"].replace(category_map)
df["category"] = df["category"].str.title()

In [7]:
df["category"].value_counts()

category
Electronics    77385
Name: count, dtype: int64

Question 7
The delivery_days column contains negative values, text entries like 'Same Day', '1-2 days', and some unrealistic values like 50 days. Clean this column to contain only valid numeric delivery days.


In [324]:
df["delivery_days"].unique()

array(['5', '6', '7', '3', '4', '2', '-1', 'Express', '1', 'Same Day',
       '1-2 days', '15', '0'], dtype=object)

In [325]:
df["delivery_days"] = df["delivery_days"].astype(str).str.strip().str.lower()

In [326]:
df["delivery_days"] = df["delivery_days"].replace({
    "same day": "0",
    "express": "1"
})

In [327]:
df["delivery_days"] = df["delivery_days"].str.extract(r"(-?\d+)")

In [328]:
df["delivery_days"] = pd.to_numeric(df["delivery_days"], errors="coerce")

In [329]:
df.loc[df["delivery_days"] < 0, "delivery_days"] = None

In [330]:
df["delivery_days"].unique()

array([ 5.,  6.,  7.,  3.,  4.,  2., nan,  1.,  0., 15.])

In [331]:
df["delivery_days"].isnull().sum()

np.int64(475)

In [332]:
df.loc[df["delivery_days"] < 0, "delivery_days"] = np.nan

df["delivery_days"] = df["delivery_days"].fillna(df["delivery_days"].median())

In [333]:
df["delivery_days"].describe()

count    77385.000000
mean         4.122634
std          1.493439
min          0.000000
25%          3.000000
50%          4.000000
75%          5.000000
max         15.000000
Name: delivery_days, dtype: float64

In [334]:
df["delivery_days"].isna().sum()

np.int64(0)

Question 8
Identify and handle duplicate transactions where the same customer, product, date, and amount appear multiple times. Some duplicates are genuine (bulk orders) while others are data errors. Develop a strategy to distinguish and handle both cases.


In [335]:
duplicate_mask = df.duplicated(
    subset=["customer_id", "product_id", "order_date", "original_price_inr"],
    keep=False
)

duplicates = df[duplicate_mask]

In [336]:
duplicate_mask = df.duplicated(
    subset=["customer_id", "product_id", "order_date", "original_price_inr"],
    keep=False
)

duplicates = df[duplicate_mask]

In [337]:
duplicates.shape

(762, 33)

In [338]:
duplicates.head()

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
86,TXN_2017_00000087,2017-01-12,CUST_2017_00009666,PROD_001927,Fitbit Tracker,Electronics,Smart Watch,Fitbit,23018.41,0.00,...,False,NaN,4.5,Delivered,1,2017,1,0.05,True,4.0
645,TXN_2017_00000646,2017-01-22,CUST_2016_00005669,PROD_001704,Lenovo iPad 8GB RAM Silver,Electronics,Tablets,Lenovo,57192.20,47.29,...,True,Republic Day Sale,4.0,Delivered,1,2017,1,0.52,True,4.3
697,TXN_2017_00000698,2017-01-31,CUST_2017_00013422,PROD_000072,Xiaomi Redmi 2 64GB Black,Electronics,Smartphones,Xiaomi,22679.83,0.00,...,False,NaN,4.5,Delivered,1,2017,1,0.19,True,3.3
983,TXN_2017_00000984,2017-01-19,CUST_2016_00001459,PROD_001683,Apple Mi Pad 8GB RAM Silver,Electronics,Tablets,Apple,54058.79,0.00,...,False,NaN,5.0,Delivered,1,2017,1,0.40,True,4.5
1028,TXN_2017_00001029,2017-01-31,CUST_2017_00012104,PROD_000230,Apple iPhone X 16GB Black,Electronics,Smartphones,Apple,127000.03,0.00,...,False,NaN,4.5,Delivered,1,2017,1,0.25,True,3.2


In [339]:
df.duplicated().sum()

np.int64(0)

In [340]:
duplicates.sort_values(
    ["customer_id","product_id","order_date"]
).head(10)

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
61242,TXN_2017_00061243,2017-11-20,CUST_2015_00000099,PROD_000176,Xiaomi Mi 5 16GB Black,Electronics,Smartphones,Xiaomi,44951.87,0.00,...,False,NaN,NaN,Delivered,11,2017,4,0.23,True,3.4
77290,TXN_2017_00061243_DUP,2017-11-20,CUST_2015_00000099,PROD_000176,Xiaomi Mi 5 16GB Black,Electronics,Smartphones,Xiaomi,44951.87,0.00,...,False,NaN,NaN,Delivered,11,2017,4,0.23,True,3.4
71278,TXN_2017_00071279,2017-12-01,CUST_2015_00000350,PROD_000199,Motorola Moto G4 Plus 32GB Black,Electronics,Smartphones,Motorola,24070.83,28.59,...,False,NaN,NaN,Delivered,12,2017,4,0.21,True,3.6
77064,TXN_2017_00071279_DUP,2017-12-01,CUST_2015_00000350,PROD_000199,Motorola Moto G4 Plus 32GB Black,Electronics,Smartphones,Motorola,24070.83,28.59,...,False,NaN,NaN,Delivered,12,2017,4,0.21,True,3.6
36172,TXN_2017_00036173,2017-07-19,CUST_2015_00000919,PROD_001605,Acer Ultrabook 4GB RAM Silver,Electronics,Laptops,Acer,134186.54,25.18,...,False,NaN,4.5,Returned,7,2017,3,1.20,True,3.4
77108,TXN_2017_00036173_DUP,2017-07-19,CUST_2015_00000919,PROD_001605,Acer Ultrabook 4GB RAM Silver,Electronics,Laptops,Acer,134186.54,25.18,...,False,NaN,4.5,Returned,7,2017,3,1.20,True,3.4
42531,TXN_2017_00042532,2017-08-13,CUST_2015_00001250,PROD_000291,Vivo V7 16GB White,Electronics,Smartphones,Vivo,32959.74,0.00,...,False,NaN,3.5,Delivered,8,2017,3,0.17,True,3.4
77345,TXN_2017_00042532_DUP,2017-08-13,CUST_2015_00001250,PROD_000291,Vivo V7 16GB White,Electronics,Smartphones,Vivo,32959.74,0.00,...,False,NaN,3.5,Delivered,8,2017,3,0.17,True,3.4
59670,TXN_2017_00059671,2017-10-03,CUST_2015_00001443,PROD_000067,Xiaomi Mi 4i 16GB White,Electronics,Smartphones,Xiaomi,18540.76,16.45,...,True,Amazon Great Indian Festival,NaN,Delivered,10,2017,4,0.25,False,4.5
77292,TXN_2017_00059671_DUP,2017-10-03,CUST_2015_00001443,PROD_000067,Xiaomi Mi 4i 16GB White,Electronics,Smartphones,Xiaomi,18540.76,16.45,...,True,Amazon Great Indian Festival,NaN,Delivered,10,2017,4,0.25,False,4.5


In [341]:
duplicates.groupby(
    ["customer_id","product_id","order_date","original_price_inr"]
).size().sort_values(ascending=False).head(10)

customer_id         product_id   order_date  original_price_inr
CUST_2015_00000099  PROD_000176  2017-11-20  44951.87              2
CUST_2015_00000350  PROD_000199  2017-12-01  24070.83              2
CUST_2015_00000919  PROD_001605  2017-07-19  134186.54             2
CUST_2015_00001250  PROD_000291  2017-08-13  32959.74              2
CUST_2015_00001443  PROD_000067  2017-10-03  18540.76              2
CUST_2015_00001460  PROD_001765  2017-08-06  26296.83              2
CUST_2015_00001537  PROD_001751  2017-12-28  21672.20              2
CUST_2015_00001562  PROD_000118  2017-08-14  232691.77             2
CUST_2015_00001734  PROD_000033  2017-02-28  189206.26             2
CUST_2015_00002004  PROD_000314  2017-10-06  27136.77              2
dtype: int64

In [342]:
dup_groups = df.groupby(
    ["customer_id","product_id","order_date","original_price_inr"]
).size().reset_index(name="count")

dup_groups = dup_groups[dup_groups["count"] > 1]

In [343]:
dup_rows = df.merge(
    dup_groups,
    on=["customer_id","product_id","order_date","original_price_inr"],
    how="inner"
)

In [344]:
dup_rows[["customer_id","product_id","quantity","count"]].head()

,customer_id,product_id,quantity,count
0,CUST_2017_00009666,PROD_001927,1,2
1,CUST_2016_00005669,PROD_001704,1,2
2,CUST_2017_00013422,PROD_000072,1,2
3,CUST_2016_00001459,PROD_001683,1,2
4,CUST_2017_00012104,PROD_000230,1,2


In [345]:
df_clean = df.drop_duplicates(
    subset=["customer_id","product_id","order_date","original_price_inr"],
    keep="first"
)

In [346]:
df_clean.duplicated(
    subset=["customer_id","product_id","order_date","original_price_inr"]
).sum()

np.int64(0)

In [347]:
df["transaction_id"].duplicated().sum()

np.int64(0)

In [348]:
df = df.drop_duplicates(subset="transaction_id", keep="first")

Question 9
The dataset contains outlier prices where some products show prices 100x higher than expected due to data entry errors (decimal point issues). Identify and correct these outliers using statistical methods and domain knowledge.


In [349]:
# ── FIX 1: Negative prices ──────────────────────────
neg_mask = df["original_price_inr"] < 0
df.loc[neg_mask, "original_price_inr"] = df.loc[neg_mask, "original_price_inr"].abs()
print(f"Negative prices fixed: {neg_mask.sum()}")

# ── FIX 2: Outliers (100x decimal error) ────────────
subcategory_caps = {
    "Smart Watch":        50000,
    "Tablets":            80000,
    "Smartphones":        100000,
    "Laptops":            200000,
    "TV & Entertainment": 200000,
    "Audio":              700000,
}

outlier_mask = df.apply(
    lambda row: row["original_price_inr"] > subcategory_caps.get(row["subcategory"], 999999),
    axis=1
)
df.loc[outlier_mask, "original_price_inr"] = (
    df.loc[outlier_mask, "original_price_inr"] / 100
).round(2)
print(f"Outliers fixed: {outlier_mask.sum()}")

# ── FIX 3: Recalculate ───────────────────────────────
df["discounted_price_inr"] = (df["original_price_inr"] * (1 - df["discount_percent"] / 100)).round(2)
df["subtotal_inr"] = (df["discounted_price_inr"] * df["quantity"]).round(2)
df["final_amount_inr"] = df["subtotal_inr"]

# ── VERIFY ───────────────────────────────────────────
print(df.groupby("subcategory", observed=True)["original_price_inr"]
      .describe()[["min","max","mean","50%"]].round(2))
print(f"\nNaN in final_amount_inr:   {df['final_amount_inr'].isna().sum()}")
print(f"Negative prices remaining: {(df['original_price_inr'] < 0).sum()}")


Negative prices fixed: 186
Outliers fixed: 18504
                        min        max      mean       50%
subcategory                                               
Audio               3009.60  439762.00  19213.81  20530.41
Laptops             2018.72  201872.33  85115.08  78859.79
Smart Watch          523.57   55597.51  23648.07  26967.64
Smartphones         1001.19  214572.30  32755.79  27182.13
TV & Entertainment  2491.54  249154.19  74196.40  55223.29
Tablets              849.51   99878.19  36097.25  30339.60

NaN in final_amount_inr:   2279
Negative prices remaining: 0


In [350]:
print(f"NaN in original_price_inr: {df['original_price_inr'].isna().sum()}")

# Check the actual NaN rows
nan_rows = df[df["discounted_price_inr"].isna()]
print(nan_rows[["product_name", "original_price_inr", "discount_percent", "discounted_price_inr"]].head(10))

NaN in original_price_inr: 2279
                        product_name  original_price_inr  discount_percent  \
17     OnePlus OnePlus 3T 16GB Black                 NaN             13.35   
47         Xiaomi Redmi 2 16GB White                 NaN              0.00   
68   Apple Galaxy Tab 4GB RAM Silver                 NaN             10.25   
98              Fitbit Watch Premium                 NaN              0.00   
178           Xiaomi Mi 5 32GB Black                 NaN              0.00   
239               Vivo V7+ 32GB Blue                 NaN             28.18   
256               Oppo F5 16GB Black                 NaN             21.73   
261              Xiaomi Watch Deluxe                 NaN             26.29   
348     Samsung Galaxy J7 16GB Black                 NaN              0.00   
373    OnePlus OnePlus 5T 16GB Black                 NaN             11.36   

     discounted_price_inr  
17                    NaN  
47                    NaN  
68                    NaN

In [351]:
# Check which subcategories these NaN prices belong to
print(df[df["original_price_inr"].isna()]["subcategory"].value_counts())

subcategory
Smartphones           1565
Laptops                210
Tablets                200
Smart Watch            150
Audio                  110
TV & Entertainment      44
Name: count, dtype: int64


In [352]:
print(df[df["original_price_inr"].isna()].head(10).to_string())

        transaction_id order_date         customer_id   product_id                     product_name     category  subcategory    brand  original_price_inr  discount_percent  discounted_price_inr  quantity  subtotal_inr  final_amount_inr customer_city customer_state customer_tier customer_spending_tier customer_age_group payment_method  delivery_days delivery_type  is_prime_member  is_festival_sale festival_name  customer_rating return_status  order_month  order_year  order_quarter  product_weight_kg  is_prime_eligible  product_rating
17   TXN_2017_00000018 2017-01-06  CUST_2017_00009783  PROD_000163    OnePlus OnePlus 3T 16GB Black  Electronics  Smartphones  OnePlus                 NaN             13.35                   NaN         1           NaN               NaN        Mumbai    Maharashtra         Metro                 Budget                55+            COD            5.0      Standard            False             False           NaN              NaN     Delivered            1  

In [353]:
# Drop rows with NaN original_price_inr
df = df.dropna(subset=["original_price_inr"]).copy()  # ← added .copy()

# Recalculate
df["discounted_price_inr"] = (df["original_price_inr"] * (1 - df["discount_percent"] / 100)).round(2)
df["subtotal_inr"] = (df["discounted_price_inr"] * df["quantity"]).round(2)
df["final_amount_inr"] = df["subtotal_inr"]




In [354]:
# Check if any legitimate products were over-corrected in cleaned files
for sub, cap in subcategory_caps.items():
    over = df[(df["subcategory"] == sub) & 
                    (df["original_price_inr"] > cap)]
    if len(over) > 0:
        print(f"\n{sub} (cap ₹{cap:,}): {len(over)} rows over")
        print(over[["product_name", "original_price_inr"]].drop_duplicates().head(5))
    else:
        print(f"\n{sub}: ✅ All within cap")


Smart Watch (cap ₹50,000): 2 rows over
               product_name  original_price_inr
23110  Xiaomi Watch Premium            53836.21
39583          Fitbit Watch            55597.51

Tablets (cap ₹80,000): 1 rows over
                      product_name  original_price_inr
6387  Samsung Slate 8GB RAM Silver            99878.19

Smartphones (cap ₹100,000): 44 rows over
                       product_name  original_price_inr
1539   Samsung Galaxy S7 64GB Black           108418.77
3205   Samsung Galaxy S7 32GB White           166440.68
4328     Apple iPhone SE 32GB White           105539.05
12395     Apple iPhone 7 64GB Black           171823.46
15300     Apple iPhone 6 64GB Black           133091.51

Laptops (cap ₹200,000): 1 rows over
                       product_name  original_price_inr
14149  Lenovo MacBook 4GB RAM Black           201872.33

TV & Entertainment (cap ₹200,000): 1 rows over
      product_name  original_price_inr
48747     LG 4K TV           249154.19

Audio: ✅ All wit

Question 10
Payment methods contain inconsistent naming: 'UPI/PhonePe/GooglePay', 'Credit Card/CREDIT_CARD/CC', 
'Cash on Delivery/COD/C.O.D'. Standardize payment method categories and create a clean categorical hierarchy.


In [355]:
# ── Standardize payment methods ────────────────────
payment_standardize = {
    # UPI variants
    "UPI": "UPI", "PhonePe": "UPI", "GooglePay": "UPI", "Google Pay": "UPI",
    "UPI/PhonePe": "UPI", "UPI/GooglePay": "UPI",

    # Credit Card variants
    "Credit Card": "Credit Card", "CREDIT_CARD": "Credit Card", "CC": "Credit Card",

    # Debit Card variants
    "Debit Card": "Debit Card", "DEBIT_CARD": "Debit Card", "DC": "Debit Card",

    # COD variants
    "Cash on Delivery": "COD", "COD": "COD", "C.O.D": "COD",

    # Others
    "Wallet": "Wallet",
    "Net Banking": "Net Banking",
    "BNPL": "BNPL"
}

df["payment_method"] = df["payment_method"].map(payment_standardize).fillna(df["payment_method"])

# ── Create categorical hierarchy ───────────────────
payment_category = {
    "UPI":          "Digital Payment",
    "Wallet":       "Digital Payment",
    "Net Banking":  "Digital Payment",
    "Credit Card":  "Card Payment",
    "Debit Card":   "Card Payment",
    "BNPL":         "Pay Later",
    "COD":          "Cash on Delivery"
}

df["payment_category"] = df["payment_method"].map(payment_category).astype("category")

# ── Verify ─────────────────────────────────────────
print(df["payment_method"].value_counts())
print(f"\nNaN in payment_category: {df['payment_category'].isna().sum()}")

payment_method
COD            45024
Credit Card    12058
Debit Card      8973
UPI             5311
Net Banking     3740
Name: count, dtype: int64

NaN in payment_category: 0


Handling Nan - in customer age group

In [356]:
df["customer_age_group"] = df["customer_age_group"].fillna("Unknown")

Checking for object columns

In [357]:
df.select_dtypes(include="object").columns

Index(['transaction_id', 'customer_id', 'product_id', 'product_name',
       'category', 'subcategory', 'brand', 'customer_city', 'customer_state',
       'customer_tier', 'customer_spending_tier', 'customer_age_group',
       'payment_method', 'delivery_type', 'festival_name', 'return_status'],
      dtype='object')

In [358]:
# ── Optimize memory: convert to category dtype ───────
cat_columns = ["category", "subcategory", "customer_tier", 
               "customer_spending_tier", "customer_age_group",
               "payment_method", "delivery_type", 
               "festival_name", "return_status"]

df[cat_columns] = df[cat_columns].astype("category")

# Verify
print(df[cat_columns].dtypes)
print(f"\nMemory usage after optimization:")
print(df.memory_usage(deep=True).sum() / 1024**2, "MB")

category                  category
subcategory               category
customer_tier             category
customer_spending_tier    category
customer_age_group        category
payment_method            category
delivery_type             category
festival_name             category
return_status             category
dtype: object

Memory usage after optimization:
40.87765979766846 MB


In [359]:
# NaN summary for all columns
nan_summary = df.isna().sum()
nan_summary = nan_summary[nan_summary > 0].sort_values(ascending=False)

print(f"Total columns with NaN: {len(nan_summary)}")
print(f"Total rows in dataset: {df.shape[0]}")
print(f"\nNaN counts and percentages:")
print(pd.DataFrame({
    "NaN Count": nan_summary,
    "Percentage": (nan_summary / df.shape[0] * 100).round(2)
}))

Total columns with NaN: 2
Total rows in dataset: 75106

NaN counts and percentages:
                 NaN Count  Percentage
festival_name        51292       68.29
customer_rating      22873       30.45


In [360]:
df.to_csv("data_cleaning_2017.csv", index=False)
print("File saved successfully!")

File saved successfully!
